In [1]:
import re

def load_poems(file_path, limit=200):
    
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    
    prompt_targets = []
    poem_lines = []
    count = 0

    # Pattern pentru a detecta titlurile sonetelor (I., II., III.)
    sonnet_title_pattern = re.compile(r'^[IVXLCDM]+\.$')

    for line in lines:
        line = line.strip()

        
        if sonnet_title_pattern.match(line):
            
            if poem_lines:
                poem_lines = [l.strip() for l in poem_lines if l.strip()]  
                if len(poem_lines) >= 6: 
                    prompt = poem_lines[1]  
                    target = poem_lines[2:6]  
                    prompt_targets.append((prompt, target))
                    count += 1
                poem_lines = [line]  
            else:
                poem_lines = [line]

        else:
            poem_lines.append(line)

        
        if count >= limit:
            break

    
    if poem_lines:
        poem_lines = [l.strip() for l in poem_lines if l.strip()]  #
        if len(poem_lines) >= 6:
            prompt = poem_lines[1]
            target = poem_lines[2:6]
            prompt_targets.append((prompt, target))

    return prompt_targets


In [5]:
from transformers import pipeline, set_seed
from transformers.utils import logging
logging.set_verbosity_error()

def generate_poetry(prompt, model_name="gpt2", max_length=50, temperature=1.0, top_p=0.9):
    #creez un pipline de generare de text
    generator = pipeline("text-generation", model=model_name,tokenizer=model_name)
    #apelez generatorul pentru a produce poezia(do_sample-pentru creativitate,temp-controleaza imprevizibilitatea, top_p-nucleu sampling(alege din cele mai probabile x%))
    response = generator(prompt, max_length=max_length, do_sample=True, temperature=temperature, top_p=top_p,truncation=True)
    #returnez doar textul generat
    return response[0]['generated_text']

In [3]:
from nltk.translate.bleu_score import sentence_bleu

def compute_bleu(reference_lines, generated_lines):
    
    total_score = 0
    count = 0
    for ref, gen in zip(reference_lines, generated_lines):
        ref_tokens = ref.strip().split()
        gen_tokens = gen.strip().split()
        #compar propozitie cu propozitie
        score = sentence_bleu([ref_tokens], gen_tokens)
        total_score += score
        count += 1
    return total_score / count if count > 0 else 0.0


In [4]:
from datasets import load_dataset



# Încarcă prompturi și ținte
prompt_targets = load_poems("sonetele_lui_shakespeare.txt",50)

# Parametri
temperatures = [0.1,0.5,0.7, 1.0]#creativitatea cuvintelor
top_ps = [0.5,0.8,0.9]#suma probabilitatilor cuvintelor<top_p


for temp in temperatures:
    for top_p in top_ps:
        print(f"\n\n=== MODEL: striki-ai/william-shakespeare-poetry, temp={temp}, top_p={top_p} ===")
        for idx, (prompt, target_lines) in enumerate(prompt_targets[:5]):
            full_prompt =  (
                "Generate the next lines of Shakespeare's original sonnet based on the given first line. "+ prompt +"\n"
                )
            gen_text = generate_poetry(
                full_prompt,
                model_name="striki-ai/william-shakespeare-poetry",
                max_length=100,
                temperature=temp,
                top_p=top_p
            )

            generated_lines = gen_text.split('\n')[1:len(target_lines)+1]  # extrage versuri generate
            bleu = compute_bleu(target_lines, generated_lines)
            result = {
                "model": "striki-ai/william-shakespeare-poetry",
                "prompt": prompt,
                "reference": target_lines,
                "generated": generated_lines,
                "bleu": bleu
            }
            print(f"Prompt: {prompt}\nGenerated: {generated_lines}\nTarget: {target_lines}\nBLEU: {bleu:.4f}")

for temp in temperatures:
    for top_p in top_ps:
        print(f"\n\n=== MODEL: gpt2, temp={temp}, top_p={top_p} ===")
        for idx, (prompt, target_lines) in enumerate(prompt_targets[:5]):
            full_prompt =  (
                "Generate the next lines of Shakespeare's original sonnet based on the given first line. "+ prompt +"\n"
                )
            gen_text = generate_poetry(
                full_prompt,
                model_name="gpt2",
                max_length=100,
                temperature=temp,
                top_p=top_p
            )

            generated_lines = gen_text.split('\n')[1:len(target_lines)+1]  # extrage versuri generate
            bleu = compute_bleu(target_lines, generated_lines)
            result = {
                "model":"gpt2",
                "prompt": prompt,
                "reference": target_lines,
                "generated": generated_lines,
                "bleu": bleu
            }
            print(f"Prompt: {prompt}\nGenerated: {generated_lines}\nTarget: {target_lines}\nBLEU: {bleu:.4f}")
               

['line', 'gutenberg_id']
{'line': 'The Song of Hiawatha is based on the legends and stories of', 'gutenberg_id': 19}


=== MODEL: striki-ai/william-shakespeare-poetry, temp=0.1, top_p=0.5 ===
Prompt: From fairest creatures we desire increase,
Generated: ['And to the fairest we desire decrease.', '', "'Tis the time of my life to write,", 'To make my love more precious,']
Target: ["That thereby beauty's rose might never die,", 'But as the riper should by time decease,', 'His tender heir might bear his memory:', 'But thou contracted to thine own bright eyes,']
BLEU: 0.0000
Prompt: When forty winters shall besiege thy brow,
Generated: ['And thou shalt not be content with the present,', 'But with the present, and with the past,', 'And with the present, and with the past,', 'And with the present, and with the past,']
Target: ["And dig deep trenches in thy beauty's field,", "Thy youth's proud livery so gazed on now,", "Will be a totter'd weed of small worth held:", 'Then being asked, where al

In [5]:
#prompt in romana model engleza c3

prompt = "Continue the poem: Afară-i toamnă, frunza 'mprăștiată,\n"
target=["Iar vântul svârlă 'n geamuri grele picuri;","Și tu citești scrisori din roase plicuri","Și într'un ceas gândești la viața toată.","Pierzându-ți timpul tău cu dulci nimicuri,"]
gen_text = generate_poetry(
                prompt,
                model_name="gpt2",
                max_length=100,
                temperature=0.7,
                top_p=0.8
            )

generated_lines = gen_text.split('\n')[1:len(target)+1] 
bleu = compute_bleu(target, generated_lines)
result = {
        "model":"gpt2",
        "prompt": prompt,
        "reference": target,
        "generated": generated_lines,
        "bleu": bleu
}

print(f"Prompt: {prompt}\nGenerated: {generated_lines}\nTarget:{target}\nBLEU: {bleu:.4f}")

Prompt: Continue the poem: Afară-i toamnă, frunza 'mprăștiată,

Generated: ['', "firă-i toamnă, frunza'mprăștiată,", '', "firă-i toamnă, frunza'mprăștiată,"]
Target:["Iar vântul svârlă 'n geamuri grele picuri;", 'Și tu citești scrisori din roase plicuri', "Și într'un ceas gândești la viața toată.", 'Pierzându-ți timpul tău cu dulci nimicuri,']
BLEU: 0.0000


In [6]:
#prompt in romana model engleza c4

prompt = "Continue the poem: Afară-i toamnă, frunza 'mprăștiată,"
target=["Iar vântul svârlă 'n geamuri grele picuri;","Și tu citești scrisori din roase plicuri","Și într'un ceas gândești la viața toată.","Pierzându-ți timpul tău cu dulci nimicuri,"]
gen_text = generate_poetry(
                prompt,
                model_name="striki-ai/william-shakespeare-poetry",
                max_length=100,
                temperature=0.7,
                top_p=0.8
            )

generated_lines = gen_text.split('\n')[1:len(target)+1]  # extrage versuri generate
bleu = compute_bleu(target, generated_lines)
result = {
        "model": "striki-ai/william-shakespeare-poetry",
        "prompt": prompt,
        "reference": target,
        "generated": generated_lines,
        "bleu": bleu
}

print(f"Prompt: {prompt}\nGenerated: {generated_lines}\nTarget:{target}\nBLEU: {bleu:.4f}")

Prompt: Continue the poem: Afară-i toamnă, frunza 'mprăștiată,
Generated: []
Target:["Iar vântul svârlă 'n geamuri grele picuri;", 'Și tu citești scrisori din roase plicuri', "Și într'un ceas gândești la viața toată.", 'Pierzându-ți timpul tău cu dulci nimicuri,']
BLEU: 0.0000


In [7]:
#generare pastel c5

prompt = "Continue the poem in the style of pastel : The sunset sky"
gen_text = generate_poetry(
                prompt,
                model_name="gpt2",
                max_length=100,
                temperature=0.7,
                top_p=0.8
            )
generated_lines = gen_text
print(generated_lines)

Continue the poem in the style of pastel : The sunset sky, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset, the sunset


In [8]:
#fine tuning

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, TextDataset, DataCollatorForLanguageModeling

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

#creez setul de date
dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="pastel_poems.txt",
    block_size=128
)

#pregatesc datele mlm=false=>modelul invata sa prezica urmatorul cuvant nu sa completeze cuvinte lipsa
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

#parametrii de antrenare
training_args = TrainingArguments(
    output_dir="./pastel-gpt2",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
)

#initializz trainerul
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()


/opt/conda/lib/python3.11/site-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'train_runtime': 26.8331, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.224, 'train_loss': 4.849401473999023, 'epoch': 3.0}


TrainOutput(global_step=6, training_loss=4.849401473999023, metrics={'train_runtime': 26.8331, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.224, 'train_loss': 4.849401473999023, 'epoch': 3.0})

In [7]:
#generare pastel c5 after fine tuning

prompt = "Continue the poem in the style of pastel : The sunset sky"
gen_text = generate_poetry(
                prompt,
                model_name="./pastel-gpt2/checkpoint-6",
                max_length=300,
                temperature=0.9,
                top_p=0.9
            )
generated_lines = gen_text
print(generated_lines)

Continue the poem in the style of pastel : The sunset sky of the moon has passed through the woods ; And the night, the night will be filled with darkness ; And the dawn of night will be filled with darkness ; And the night will be filled with darkness ; And the moon will be filled with darkness ; And the day will be filled with darkness ; And the day will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled with darkness ; And the night will be filled w

In [8]:
#fine tuning

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, TextDataset, DataCollatorForLanguageModeling

tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
model = GPT2LMHeadModel.from_pretrained("distilgpt2")

dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="pastel_poems.txt",
    block_size=128
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

training_args = TrainingArguments(
    output_dir="./pastel2-gpt2",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()


/opt/conda/lib/python3.11/site-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'train_runtime': 22.4401, 'train_samples_per_second': 0.401, 'train_steps_per_second': 0.267, 'train_loss': 4.677359263102214, 'epoch': 3.0}


TrainOutput(global_step=6, training_loss=4.677359263102214, metrics={'train_runtime': 22.4401, 'train_samples_per_second': 0.401, 'train_steps_per_second': 0.267, 'train_loss': 4.677359263102214, 'epoch': 3.0})

In [12]:
#generare pastel c5 after fine tuning cu distilgpt2-mai rapid

prompt = "Continue the poem in the style of pastel : The sunset sky"
gen_text = generate_poetry(
                prompt,
                model_name="./pastel2-gpt2/checkpoint-6",
                max_length=200,
                temperature=0.9,
                top_p=0.9
            )
generated_lines = gen_text
print(generated_lines)

Continue the poem in the style of pastel : The sunset sky is shining with light, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the sky, and the moon rises to the


In [12]:
c.1 care sunt diferentele de calitate intre textele generate cu cele doua tipuri de LLM-uri?

    -LLM pre-antrenat general(gpt2)
        INFLUENTA PARAMETRILOR:
        -Temperature:
            *valori mici=>completari banale dar coerente
            *valori mari=>completari creative, uneori incoerente
        -Top-p:
            *0.8,0.9 sunt ok
            *1.0 duce la generare prea random
        -Tokenizer(BPE-Byte pair encoding):
            *functioneaza bine pe engleza
            *imparte textul in cele mai frecvente secvente de caractere

        LIMBAJ POETIC: rigid
        COERENTA: variabila

    -LLM adaptat(striki-ai/william-shakespeare-poetry)
        INFLUENTA PARAMETRILOR:
        -Temperature:
            *valori mici=>completari banale dar coerente
            *valori mari=>completari creative
        -Top-p:
            *0.8,0.9 sunt ok
            *1.0 duce la generare prea random
        -Tokenizer(BPE-Byte pair encoding):
            *functioneaza bine pe engleza
            *imparte textul in cele mai frecvente secvente de caractere

        LIMBAJ POETIC: stilizat
        COERENTA: in mare parte coerent

c.2 ce se intampla daca versurile din prompt sunt in limba engleza?
    -ambele LLM-uri se descurca bine

c.3 ce se intampla daca versurile din prompt sunt in limba romana?
    -nu genereaza ceva coerent dar foloseste cuvinte relativ corecte din romana

c.4 ce se intampla daca versurile din prompt sunt in limba romana si corpusul de antrenare este in limba engleza?
    -nu genereaza nimic

c.5 cum se poate "personaliza" LLM pentru a genera versuri in stil de pastel (cu accent pe frumusetea naturii)?
    -am adaptat parametrii cu valori mai mari pentru o creativitate imbunatatita
    -am schimbat promptul
    -fine-tune pe cateva poezii tip pastel


IndentationError: unexpected indent (1119686651.py, line 3)